# 02 - Preprocessing et Feature Engineering

## Objectifs

Préparer les données pour la modélisation en appliquant dans l'ordre :
1. Feature engineering (création de variables dérivées)
2. Split temporel (obligatoire - pas de split aléatoire)
3. Encodage des variables catégorielles
4. Gestion des valeurs manquantes
5. Évaluation des différentes stratégies de gestion du déséquilibre

À la fin de ce notebook, on dispose de X_train, X_test, y_train, y_test prêts pour la modélisation.

In [11]:
import sys
from pathlib import Path
sys.path.append('..')
PARQUETS_DIR = Path('../data/parquets')

import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
from src.data_loader import load_baf
from src.preprocessing import (
    temporal_split, encode_categoricals, engineer_features,
    handle_imbalance, preprocess_pipeline
)

print("Setup terminé.")


Setup terminé.


## 1. Chargement et inspection initiale

In [12]:
df = load_baf('../data/', variant='Base')
print(f"Forme initiale : {df.shape}")

Chargement de Base.csv...

Valeurs manquantes corrigées (codage négatif → NaN) :
  intended_balcon_amount              :  742,523 (74.3%) (toutes négatives)
  prev_address_months_count           :  712,920 (71.3%) (-1 uniquement)
  bank_months_count                   :  253,635 (25.4%) (-1 uniquement)
  current_address_months_count        :    4,254 (0.4%) (-1 uniquement)
  session_length_in_minutes           :    2,015 (0.2%) (-1 uniquement)
  device_distinct_emails_8w           :      359 (0.0%) (-1 uniquement)
Dataset chargé : 1,000,000 lignes × 32 colonnes
Taux de fraude : 1.10%
Forme initiale : (1000000, 32)


## 2. Feature engineering

On crée 6 features dérivées identifiées comme pertinentes lors de l'EDA :

| Feature | Calcul | Signal capturé |
|---------|--------|----------------|
| `velocity_ratio_6h_24h` | velocity_6h / (velocity_24h + 1) | Concentration temporelle 6h vs 24h |
| `velocity_ratio_6h_4w` | velocity_6h / (velocity_4w + 1) | Concentration temporelle 6h vs 4w |
| `device_ever_fraud` | (device_fraud_count > 0) | Flag binaire - device déjà flagué |
| `both_phones_valid` | (phone_home_valid & phone_mobile_valid) | Cohérence téléphonique |
| `no_phone_valid` | (!phone_home_valid & !phone_mobile_valid) | Absence totale de validation |
| `risk_flags_count` | Compteur de 4 flags suspects | Score agrégé de risque |

In [13]:
df_eng = engineer_features(df)
new_features = [c for c in df_eng.columns if c not in df.columns]
print(f"Features ajoutées ({len(new_features)}) : {new_features}")
df_eng[new_features].head()

Features ajoutées (6) : ['velocity_ratio_6h_24h', 'velocity_ratio_6h_4w', 'device_ever_fraud', 'both_phones_valid', 'no_phone_valid', 'risk_flags_count']


,velocity_ratio_6h_24h,velocity_ratio_6h_4w,device_ever_fraud,both_phones_valid,no_phone_valid,risk_flags_count
0,1.667869,1.942144,0,0,0,2
1,1.605096,1.552045,0,1,0,1
2,0.817007,0.746047,0,0,0,2
3,2.136065,2.416878,0,0,0,2
4,1.483208,1.279342,0,1,0,0


In [14]:
X_train, X_test, y_train, y_test = temporal_split(df_eng)

print(f"Train : {len(X_train):,} lignes (mois 0-5)")
print(f"Test  : {len(X_test):,} lignes (mois 6-7)")
print()
print(f"Taux de fraude TRAIN : {y_train.mean():.2%}")
print(f"Taux de fraude TEST  : {y_test.mean():.2%}")
print()
print(f"Ratio fraude TEST/TRAIN : {y_test.mean() / y_train.mean():.2f}")

Train : 794,989 lignes (mois 0-5)
Test  : 205,011 lignes (mois 6-7)

Taux de fraude TRAIN : 1.03%
Taux de fraude TEST  : 1.40%

Ratio fraude TEST/TRAIN : 1.37


**Observation importante :** Le taux de fraude est plus élevé dans le test set que dans le train set.
C'est la dérive temporelle en action. Un modèle entraîné sur le train sera donc **sous-calibré**.

## 4. Encodage des variables catégorielles

In [15]:
# LabelEncoder pour les modèles à base d'arbres (XGBoost, RandomForest)
X_train_enc, X_test_enc = encode_categoricals(X_train, X_test, method='label')
print(f"Train encodé : {X_train_enc.shape}")
print(f"Test encodé  : {X_test_enc.shape}")
X_train_enc.head(3)

Train encodé : (794989, 36)
Test encodé  : (205011, 36)


,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,velocity_6h,velocity_24h,velocity_4w,bank_branch_count_8w,date_of_birth_distinct_emails_4w,employment_status,credit_risk_score,email_is_free,housing_status,phone_home_valid,phone_mobile_valid,bank_months_count,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,velocity_ratio_6h_24h,velocity_ratio_6h_4w,device_ever_fraud,both_phones_valid,no_phone_valid,risk_flags_count
0,0.3,0.986506,NaN,25.0,40,0.006735,102.453711,0,1059,13096.035018,7850.955007,6742.080561,5,5,1,163,1,2,0,1,9.0,0,1500.0,0,0,16.224843,0,1,1.0,0,1.667869,1.942144,0,0,0,2
1,0.8,0.617426,NaN,89.0,20,0.010095,NaN,3,1658,9223.283431,5745.251481,5941.664859,3,18,0,154,1,2,1,1,2.0,0,1500.0,0,0,3.363854,2,1,1.0,0,1.605096,1.552045,0,1,0,1
2,0.8,0.996707,9.0,14.0,40,0.012316,NaN,1,1095,4471.472149,5471.988958,5992.555113,15,11,0,89,1,2,0,1,30.0,0,200.0,0,0,22.730559,3,0,1.0,0,0.817007,0.746047,0,0,0,2


## 5. Gestion des valeurs manquantes

In [16]:
# Pour les modèles à base d'arbres, on impute par -999 (valeur sentinelle)
# Les arbres apprennent naturellement à séparer cette valeur des autres
X_train_enc = X_train_enc.fillna(-999)
X_test_enc = X_test_enc.fillna(-999)

# Pour les modèles DL ou linéaires, on utiliserait plutôt SimpleImputer(strategy='median')
print(f"NaN restants dans train : {X_train_enc.isna().sum().sum()}")
print(f"NaN restants dans test  : {X_test_enc.isna().sum().sum()}")

NaN restants dans train : 0
NaN restants dans test  : 0


## 6. Comparaison des stratégies anti-déséquilibre

On évalue 3 stratégies sur les mêmes données :

In [17]:
# Distribution AVANT
print("AVANT resampling :")
print(y_train.value_counts())
print(f"Ratio : 1:{int((y_train==0).sum() / (y_train==1).sum())}")
print()

# Stratégie 1 — SMOTE (oversampling synthétique vers 10%)
X_smote, y_smote = handle_imbalance(X_train_enc, y_train,
                                     method='smote', sampling_strategy=0.1)
print("APRÈS SMOTE (sampling_strategy=0.1) :")
print(y_smote.value_counts())
print(f"Ratio : 1:{int((y_smote==0).sum() / (y_smote==1).sum())}")
print()

# Stratégie 2 — UnderSampling vers 10%
X_under, y_under = handle_imbalance(X_train_enc, y_train,
                                      method='undersample', sampling_strategy=0.1)
print("APRÈS UnderSampling (sampling_strategy=0.1) :")
print(y_under.value_counts())
print(f"Ratio : 1:{int((y_under==0).sum() / (y_under==1).sum())}")

AVANT resampling :
fraud_bool
0    786838
1      8151
Name: count, dtype: int64
Ratio : 1:96

APRÈS SMOTE (sampling_strategy=0.1) :
fraud_bool
0    786838
1     78683
Name: count, dtype: int64
Ratio : 1:10

APRÈS UnderSampling (sampling_strategy=0.1) :
fraud_bool
0    81510
1     8151
Name: count, dtype: int64
Ratio : 1:10


**Interprétation des stratégies :**

| Stratégie | Volume train | Avantage | Inconvénient |
|-----------|--------------|----------|--------------|
| **None** (class_weight) | 786k (inchangé) | Pas de manipulation des données, scientifiquement propre | Le modèle "voit" peu de fraudes |
| **SMOTE** | 815k (oversampling) | Plus d'exemples de fraude à apprendre | Exemples synthétiques pas toujours réalistes |
| **UnderSample** | 81k (massive réduction) | Train rapide, équilibré | Perte d'information sur les légitimes |


## 7. Pipeline complet

In [18]:
# Le pipeline encapsule toutes les étapes en une seule fonction
X_train_final, X_test_final, y_train_final, y_test_final = preprocess_pipeline(
    df,
    feature_engineering=True,
    encoding_method='label',
    imbalance_method='none',  
)

print("Pipeline appliqué.")
print(f"X_train : {X_train_final.shape}, X_test : {X_test_final.shape}")
print(f"y_train : {y_train_final.shape}, y_test : {y_test_final.shape}")

Colonnes supprimées (constantes / forcées) : ['device_ever_fraud', 'device_fraud_count']
Pipeline appliqué.
X_train : (794989, 40), X_test : (205011, 40)
y_train : (794989,), y_test : (205011,)


## 8. Sauvegarde pour les notebooks suivants

In [19]:
# Sauvegarde au format Parquet (plus rapide que CSV)
X_train_final.to_parquet(PARQUETS_DIR / 'X_train.parquet')
X_test_final.to_parquet(PARQUETS_DIR / 'X_test.parquet')
y_train_final.to_frame().to_parquet(PARQUETS_DIR / 'y_train.parquet')
y_test_final.to_frame().to_parquet(PARQUETS_DIR / 'y_test.parquet')

print(f"Données préprocessées sauvegardées dans {PARQUETS_DIR}/")

Données préprocessées sauvegardées dans ..\data\parquets/


## Conclusion

Le preprocessing est terminé. Les décisions clés :

1.  **6 features dérivées** créées sur la base des insights de l'EDA
2.  **Split temporel** mois 0-5 / 6-7 - pas de data leakage
3.  **LabelEncoder** pour compatibilité avec les modèles à base d'arbres
4.  **Imputation -999** comme valeur sentinelle (apprise par les arbres)
5.  **Pas de SMOTE** par défaut - `scale_pos_weight` est plus propre

Les notebooks suivants chargent directement les fichiers Parquet et passent à la modélisation.